# A6 network visualization — core

Adapted from University of Michigan EECS 498/598, Deep Learning for Computer Vision (Winter 2022), by Justin Johnson and course staff.

**Static validation only; not executed.** Enable the run switches explicitly; generated files go to `core_results/`.

In [ ]:
from pathlib import Path
import os
import sys
import torch
import matplotlib.pyplot as plt
candidates = []
for base in (Path.cwd(), *Path.cwd().parents,
             Path('/content/drive/MyDrive/umich-dlcv-wi2022')):
    candidates.extend((base, base / 'assignments' / 'a6'))
ASSIGNMENT_DIR = next((p.resolve() for p in candidates if (p / 'network_visualization_core.py').is_file()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError('Open from the project/assignment directory, or mount the canonical Colab Drive project first.')
loaded_helpers = sys.modules.get('eecs598')
if loaded_helpers is not None:
    loaded_path = Path(loaded_helpers.__file__).resolve().parent
    if loaded_path != (ASSIGNMENT_DIR / 'eecs598').resolve():
        raise RuntimeError('Restart the kernel before switching assignments: eecs598 is already loaded from another assignment.')
sys.path.insert(0, str(ASSIGNMENT_DIR))
os.chdir(ASSIGNMENT_DIR)
from eecs598.utils import reset_seed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = ASSIGNMENT_DIR / 'core_results' / 'network_visualization'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_OVERFIT = False
ALLOW_DOWNLOAD = False
reset_seed(0)
plt.rcParams['image.cmap'] = 'gray'
print('Outputs:', OUTPUT_DIR)

In [ ]:
import random
import numpy as np
import torchvision
from PIL import Image
from a6_helper_core import (preprocess, deprocess, load_imagenet_val, jitter, blur_image,
                           SQUEEZENET_MEAN, SQUEEZENET_STD)
from network_visualization_core import compute_saliency_maps, make_adversarial_attack, class_visualization_step

## Pretrained model and ImageNet samples
Enable `RUN_EVALUATION` to generate saliency maps, an adversarial example, and class visualizations. SqueezeNet ImageNet weights may be downloaded by torchvision when first loaded.

In [ ]:
if RUN_EVALUATION:
    from urllib.request import urlretrieve
    import hashlib
    data_path = ASSIGNMENT_DIR / 'datasets' / 'imagenet_val_25.npz'
    if not data_path.is_file():
        if not ALLOW_DOWNLOAD:
            raise FileNotFoundError(f'Provide {data_path}, or enable ALLOW_DOWNLOAD.')
        data_path.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve('https://cs231n.stanford.edu/imagenet_val_25.npz', data_path)
    assert hashlib.sha256(data_path.read_bytes()).hexdigest() == '35141609d22457c791e8fbe10c2d3b09ed0d6a04b669173e9fbebb0b4915cd3c'
    model = torchvision.models.squeezenet1_1(weights=torchvision.models.SqueezeNet1_1_Weights.DEFAULT).to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    X, y, class_names = load_imagenet_val(num=5, path=str(data_path))

## Saliency maps

In [ ]:
if RUN_EVALUATION:
    def show_saliency_maps(X, y):
        X_tensor = torch.cat([preprocess(Image.fromarray(x)) for x in X], dim=0).to(device=DEVICE)
        y_tensor = torch.tensor(y, device=DEVICE)
        saliency = compute_saliency_maps(X_tensor, y_tensor, model)
        saliency = saliency.to('cpu').numpy()
        N = X.shape[0]
        for i in range(N):
            plt.subplot(2, N, i + 1)
            plt.imshow(X[i])
            plt.axis('off')
            plt.title(class_names[y[i]])
            plt.subplot(2, N, N + i + 1)
            plt.imshow(saliency[i], cmap=plt.cm.hot)
            plt.axis('off')
            plt.gcf().set_size_inches(12, 5)
        plt.savefig(os.path.join(str(OUTPUT_DIR), 'saliency_maps_results.jpg'))
        plt.show()
    show_saliency_maps(X, y)

## Targeted attack

In [ ]:
if RUN_EVALUATION:
    idx = 0
    target_y = 6
    X_tensor = torch.cat([preprocess(Image.fromarray(x)) for x in X], dim=0).to(device=DEVICE)
    X_adv = make_adversarial_attack(X_tensor[idx:idx + 1], target_y, model, max_iter=100)
    scores = model(X_adv)
    assert target_y == scores.data.max(1)[1][0].item(), 'The model is not fooled!'

## Adversarial differences

In [ ]:
if RUN_EVALUATION:
    from a6_helper_core import deprocess
    X_adv = X_adv.to('cpu')
    X_adv_np = deprocess(X_adv.clone())
    X_adv_np = np.asarray(X_adv_np).astype(np.uint8)
    plt.subplot(1, 4, 1)
    plt.imshow(X[idx])
    plt.title(class_names[y[idx]])
    plt.axis('off')
    plt.subplot(1, 4, 2)
    plt.imshow(X_adv_np)
    plt.title(class_names[target_y])
    plt.axis('off')
    plt.subplot(1, 4, 3)
    X_pre = preprocess(Image.fromarray(X[idx]))
    diff = np.asarray(deprocess(X_adv - X_pre, should_rescale=False))
    plt.imshow(diff)
    plt.title('Difference')
    plt.axis('off')
    plt.subplot(1, 4, 4)
    diff = np.asarray(deprocess(10 * (X_adv - X_pre), should_rescale=False))
    plt.imshow(diff)
    plt.title('Magnified difference (10x)')
    plt.axis('off')
    plt.gcf().set_size_inches(12, 5)
    plt.savefig(os.path.join(str(OUTPUT_DIR), 'adversarial_attacks_results.jpg'))
    plt.show()

## Class score maximization

In [ ]:
def create_class_visualization(target_y, model, class_names, device='cpu', save_fig=False, **kwargs):
    num_iterations = kwargs.pop('num_iterations', 100)
    blur_every = kwargs.pop('blur_every', 10)
    max_jitter = kwargs.pop('max_jitter', 16)
    show_every = kwargs.pop('show_every', 25)
    img = torch.randn((1, 3, 224, 224), device=device).mul_(1.0).requires_grad_()
    for t in range(num_iterations):
        ox, oy = (random.randint(0, max_jitter), random.randint(0, max_jitter))
        img.data.copy_(jitter(img.data, ox, oy))
        img = class_visualization_step(img, target_y, model)
        img.data.copy_(jitter(img.data, -ox, -oy))
        for c in range(3):
            lo = float(-SQUEEZENET_MEAN[c] / SQUEEZENET_STD[c])
            hi = float((1.0 - SQUEEZENET_MEAN[c]) / SQUEEZENET_STD[c])
            img.data[:, c].clamp_(min=lo, max=hi)
        if t % blur_every == 0:
            blur_image(img.data, sigma=0.5)
        if t == 0 or (t + 1) % show_every == 0 or t == num_iterations - 1:
            plt.imshow(deprocess(img.data.clone().cpu()))
            class_name = class_names[target_y]
            plt.title('%s\nIteration %d / %d' % (class_name, t + 1, num_iterations))
            plt.gcf().set_size_inches(4, 4)
            plt.axis('off')
            if save_fig:
                plt.savefig(os.path.join(str(OUTPUT_DIR), 'class_viz_result.jpg'))
            plt.show()
    return deprocess(img.data.cpu())

In [ ]:
if RUN_EVALUATION:
    reset_seed(0)
    class_image = create_class_visualization(76, model, class_names, device=DEVICE, save_fig=True)
    random_target = random.randint(0, 999)
    random_image = create_class_visualization(random_target, model, class_names, device=DEVICE)
    random_image.save(OUTPUT_DIR / f'class_{random_target}.jpg')